In [0]:
class Silver_drivers_qualifying():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,table,drivers_df,constructors_df,races_df):#
         self.table=table
         self.drivers_df=drivers_df # intializing the broadcasted dataframe
         self.constructors_df=constructors_df  # intializing the broadcasted dataframe
         self.races_df=races_df  # intializing the broadcasted dataframe
    
    
    def max_watermark_value(self):
        from pyspark.sql.functions import max,col

        #fetching max_ingestion_date from gold layer table drivers_qualifying
        if spark.catalog.tableExists("formula1_race.silver.drivers_qualifying"):
           qualifying_max_ingestion_date =spark.read.table('formula1_race.silver.drivers_qualifying').agg(max(col('qualifying_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
           if qualifying_max_ingestion_date is None:
               qualifying_max_ingestion_date='1900-01-01 00:00:00'
        else:
            qualifying_max_ingestion_date='1900-01-01 00:00:00'
        #printing last water mark value 
        print(f"results_max_ingestion_date:{qualifying_max_ingestion_date}")
        return qualifying_max_ingestion_date
        
    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        qualifying_max_ingestion_date=list_max_ingest

        #fetching incremental qualifying data and printing the count of records
        incr_qualifying_df= (spark.read.table('formula1_race.silver.qualifying')
                        .filter(col('qualifying_ingestion_date')>qualifying_max_ingestion_date)
                            )
        print("incr_qualifying_df")
        display(incr_qualifying_df.select(count("*")))

        # races_df=spark.read.table('formula1_race.silver.races')
        # constructors_df=spark.read.table('formula1_race.silver.constructors')
        # drivers_df=spark.read.table('formula1_race.silver.drivers')
        # passing all requiried tables for join as list
        read_df_list=[self.races_df,self.constructors_df,self.drivers_df,incr_qualifying_df]
        return read_df_list

    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,broadcast,expr
        races_df=read_df_list[0]
        constructors_df=read_df_list[1]
        drivers_df=read_df_list[2]
        qualifying_df=read_df_list[3]
        #performing join operation only on  incremental qualifying data with all other tables
        drivers_qualifying_df= (qualifying_df
                .join(races_df,qualifying_df["race_id"] == races_df["race_id"],'inner')
                .join(drivers_df,["driver_id"],'inner')
                .join(constructors_df,["constructor_id" ],'inner')
                .selectExpr("race_year","race_name","race_date","race_time","round as race_round","driver_name","driver_nationality","constructor_team","constructor_nationality","q1 as qualifying_q1"," q2 as qualifying_q2","q3 as qualifying_q3","position as qualifying_position","qualifying_ingestion_date")
                             )
        print("detailed execution plain")
        drivers_qualifying_df.explain(True)
        
        return drivers_qualifying_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"formula1_race.silver.{self.table}"))
        print("final table record count")
        display(spark.sql(f"select count(*) from formula1_race.silver.{self.table}"))
        print("Data write into silver drivers_qualifying table is Done")
    
       
        
    def process(self):
        print("Started Silver-ingestion-drivers_qualifying  in ran....")
        qualifying_watermark=self.max_watermark_value() #return last water mark value
        read_df_list=self.read_input(qualifying_watermark)#return all required tables as list 
        apply_tran_df=self.apply_transformations(read_df_list) #return joined data
        self.write_output(apply_tran_df)#write data into gold table
        return qualifying_watermark #return last water mark value for other tables aggregations(season_summary,champions_list,race_wise_analysis etc...)

In [0]:
# Silver_drivers_qualifying_instance = Silver_drivers_qualifying("drivers_qualifying")
# drivers_qualifying_max_ingestion=Silver_drivers_qualifying_instance .process()
# print("Successfully Silver_drivers_qualifying is ran")